In [28]:
import nflreadpy as nfl
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
import joblib


In [29]:
pbp = nfl.load_pbp([2023,2024,2025])
pbp = pbp.to_pandas()

### Preparing Data

After experimenting with different feature combinations, I selected the following seven pre-play features because they provided the most useful information for predicting play type:

- Down
- Yards to go
- Field position
- Score differential
- Quarter
- Shotgun formation
- Time remaining in the quarter

These features also correspond to the information a person would naturally consider when anticipating a play call. This made them useful not only for prediction, but also for building an application where users can understand and interact with the model's inputs.

In [30]:
df = pbp.copy()

df = df[df["play_type"].isin(["run", "pass"])]
features = [
    "down",
    "ydstogo",
    "yardline_100",
    "score_differential",
    "qtr",
    "shotgun",
    "quarter_seconds_remaining"
]

target = "play_type"


df = df[features + [target]].dropna()


X = df[features]
y = df[target]

y = y.map({
    "run": 0,
    "pass": 1
})




X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


### Data Distribution

Before testing the models, I investigated the ratio of run to pass plays to establish a baseline for model performance. I also suspected that formation information would be influential, so I investigated the relationship between formation and play selection.

In [31]:
print(df["play_type"].value_counts(normalize=True))
pd.crosstab(
    df["shotgun"],
    df["play_type"],
    normalize="index"
)

play_type
pass    0.57381
run     0.42619
Name: proportion, dtype: float64


play_type,pass,run
shotgun,,
0.0,0.292674,0.707326
1.0,0.696334,0.303666


### Model Selection

After experimenting with multiple machine learning models, I ultimately chose to implement a Random Forest model because it retained strong performance with fewer features. This made it a good fit for the application, where I wanted to keep the user interface simple while maintaining predictive performance.

In [32]:

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

accuracy = accuracy_score(y_test, preds)


print("Accuracy:", accuracy)

train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)

print("Train: " + str(train_acc))


print(confusion_matrix(y_test, preds,normalize="true"))

Accuracy: 0.7191075514874142
Train: 0.7861791137279207
[[0.62444296 0.37555704]
 [0.21008333 0.78991667]]


In [33]:
joblib.dump(model, "model.pkl",compress =3)

['model.pkl']